# SENTRIX: eBPF-Driven DRL & TCN Intrusion Prevention System
### Universal Training Pipeline for Kaggle, Google Colab, and Edge PCs

This notebook allows you to train the **Temporal Convolutional Network (TCN)** threat classifier and **Deep Reinforcement Learning (PPO)** active mitigation policy on the combined **CIC-IDS 2017 & 2018** dataset.

**Supported Environments:**
* **Kaggle Notebooks:** Full GPU acceleration (NVIDIA T4 x2 / P100), automatic dataset path discovery across `/kaggle/input`, output written to `/kaggle/working`.
* **Google Colab:** GPU/TPU acceleration with automated one-click artifact download.
* **Edge PCs & Local Workstations:** Raspberry Pi 4/5, NVIDIA Jetson, Intel NUC, or low-RAM CPU with memory-bounded streaming and 50x vectorized latent precomputation.

## 1. Hardware & Environment Diagnostics
Inspect host environment, CPU cores, memory limits, and available accelerators (CUDA, Apple MPS, or CPU).

In [ ]:
import os
import sys
import platform
import torch

print("=" * 65)
print(" SENTRIX IPS: HARDWARE & ENVIRONMENT CHECK")
print("=" * 65)

is_kaggle = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
is_colab = "google.colab" in sys.modules or os.path.exists("/content")
env_str = "Kaggle" if is_kaggle else ("Google Colab" if is_colab else "Local / Edge PC")

print(f" Environment:        {env_str}")
print(f" Operating System:   {platform.system()} ({platform.machine()})")
print(f" CPU Cores:          {os.cpu_count()}")

if torch.cuda.is_available():
    print(f" CUDA Accelerator:   {torch.cuda.get_device_name(0)} ({torch.cuda.device_count()} GPU(s))")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print(" Accelerator:        Apple Silicon MPS")
else:
    print(" Accelerator:        CPU (Multi-threaded execution optimized for Edge)")

print(f" PyTorch Version:    {torch.__version__}")
print("=" * 65)

## 2. Install Required Dependencies
Ensure all prerequisite Python libraries (PyTorch, Gymnasium, scikit-learn, PyYAML, joblib) are installed.

In [ ]:
!pip install -q pyyaml gymnasium scikit-learn matplotlib seaborn joblib tqdm

## 3. Auto-Detect Dataset Locations
The system scans `/kaggle/input/**/` or local directories for raw flow CSVs or preprocessed `.npz` files.

In [ ]:
from engine.platform_utils import (
    print_platform_diagnostics,
    resolve_data_dir,
    resolve_raw_dirs,
    get_profile_settings,
)

print_platform_diagnostics()

data_dir = resolve_data_dir()
print(f"\nResolved Preprocessed Data Directory: {data_dir}")

raw_dirs = resolve_raw_dirs(cfg_dirs=["CICIDS 2017", "CICIDS 2018"])
print(f"Resolved Raw Dataset Directories:      {raw_dirs}")

## 4. Data Ingestion & Reservoir Sampling

Choose an execution profile:
* **`--profile standard`:** For Kaggle / Colab Cloud GPUs (30,000 samples per class across 2017 & 2018).
* **`--profile edge` (or `--edge`):** For Edge PCs or low RAM (5,000 samples per class, bounded <200MB memory).

*Note: If preprocessed `.npz` files already exist in `data/processed/` or Kaggle inputs, this step can be skipped.*

In [ ]:
import os

train_npz = os.path.join(data_dir, "train_data.npz")
if not os.path.exists(train_npz):
    print("Preprocessed dataset not found. Running streaming ingestion...")
    # For Edge PCs, use: !python scripts/01_preprocess_data.py --edge
    !python scripts/01_preprocess_data.py --profile standard
else:
    print(f"Preprocessed dataset verified at {train_npz}. Skipping raw ingestion.")

In [ ]:
import numpy as np
import json

meta_file = os.path.join(data_dir, "metadata.json")
if os.path.exists(meta_file):
    with open(meta_file, "r") as f:
        meta = json.load(f)
    print("Dataset Metadata:", json.dumps(meta, indent=2))

if os.path.exists(train_npz):
    train_data = np.load(train_npz)
    print(f"X_train shape: {train_data['X'].shape}")
    print(f"Class distribution in train set: {np.bincount(train_data['y'])}")

## 5. Train TCN Threat Detector
Train the dilated causal Temporal Convolutional Network with automatic hardware acceleration and Mixed Precision (AMP).

In [ ]:
# For Edge PC: use !python scripts/02_train_tcn.py --edge
# For Kaggle / Colab GPU: use !python scripts/02_train_tcn.py --device auto --amp
!python scripts/02_train_tcn.py --device auto --amp

## 6. Train PPO Deep Reinforcement Learning Policy
Train the DRL mitigation agent with the eBPF simulator.
Precomputed TCN latent embeddings accelerate rollouts by **50x-100x** on Edge PCs and Kaggle.

In [ ]:
# For Edge PC: use !python scripts/03_train_drl.py --edge
# For Kaggle / Colab GPU: use !python scripts/03_train_drl.py --device auto
!python scripts/03_train_drl.py --device auto

## 7. Comprehensive System Evaluation
Evaluate TCN intrusion detection accuracy, macro F1, and end-to-end eBPF-RL mitigation latency and false positive rates.

In [ ]:
!python scripts/05_evaluate.py --device auto

## 8. Display Evaluation Results & Plots

In [ ]:
import matplotlib.pyplot as plt
import glob
from PIL import Image

for img_path in sorted(glob.glob("results/*.png")):
    print(f"Displaying: {img_path}")
    img = Image.open(img_path)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

## 9. Export Optimized Edge Models (TorchScript & INT8 Quantization)
Compiles the trained models into lightweight formats for deployment on Raspberry Pi, NVIDIA Jetson, or Edge gateways.

In [ ]:
!python scripts/export_edge_model.py

## 10. Archive and Export Trained Artifacts
Bundle all trained models, evaluation figures, and edge deployment packages into a single ZIP archive.

* On **Kaggle**: The file `ips_trained_artifacts.zip` will appear under `/kaggle/working` and can be downloaded from the **Output** tab in the right sidebar.
* On **Google Colab**: The download will trigger automatically in your browser.

In [ ]:
!zip -r ips_trained_artifacts.zip checkpoints/ results/ artifacts/ data/processed/metadata.json

try:
    from google.colab import files
    files.download("ips_trained_artifacts.zip")
    print("Download initiated via Google Colab.")
except ImportError:
    print("Artifacts archived into ips_trained_artifacts.zip.")
    if os.path.exists("/kaggle/working"):
        print("On Kaggle: Download 'ips_trained_artifacts.zip' from the Output pane on the right sidebar.")